# Text, tables and facts across markets

Text, HTML tables and tagged facts where the original carries them. Values stay
the filed strings: no accounting normalisation.

## Identify your SEC requests

The SEC asks automated clients for a name or organization and a contact email,
separated by a space. For example, `Alex Morgan alex@example.com` or
`Example Research filings@example.com`. These are examples: use your own name
and a working email. No SEC account or API key is needed.

In the next cell, uncomment the `os.environ[...]` line and replace the example
with your details. If you already set this environment variable in your
terminal, leave the line commented. The label is sent to the SEC with each
request. See the [setup guide](../../docs/getting-started.md#sec-request-identity).

In [1]:
import os
from collections import Counter

from opencorpfilings import Company, provider_info, search_companies

# Uncomment this line and replace the example with your own name and contact email.
# os.environ["OPENCORPFILINGS_SEC_USER_AGENT"] = "Alex Morgan alex@example.com"
assert os.getenv("OPENCORPFILINGS_SEC_USER_AGENT"), "Uncomment the SEC identity example above and replace it with your own details, or set OPENCORPFILINGS_SEC_USER_AGENT in your environment."

UK: J Sainsbury files its accounts in inline XBRL (many large companies file a PDF instead):

In [2]:
accounts = Company("00185647", market="UK").filings(form="annual").latest()
frame = accounts.facts_to_pandas()
print(len(frame), "facts")
frame[frame["concept_raw"].str.contains("Revenue")][["concept_raw", "value", "unit", "period_end"]].head()

762 facts


,concept_raw,value,unit,period_end
16,ifrs-full:Revenue,"33,6 4 7",iso4217:GBP,2026-02-28
17,ifrs-full:Revenue,—,iso4217:GBP,2026-02-28
18,ifrs-full:Revenue,"3 3,6 47",iso4217:GBP,2026-02-28
19,ifrs-full:Revenue,"32 ,7 7 2",iso4217:GBP,2025-03-01
20,ifrs-full:Revenue,—,iso4217:GBP,2025-03-01


Values are the text exactly as filed. This filing's iXBRL was generated from a PDF,
so its text has spaces inside numbers (`33,6 4 7`). The source's format and scale
travel with each fact, so you can apply them yourself: here `scale="6"` means
£33,647 million.

In [3]:
fact = next(f for f in accounts.facts if f.concept_raw == "ifrs-full:Revenue")
fact.value, {key: fact.provider_metadata["attributes"].get(key) for key in ("format", "scale", "decimals")}

('33,6 4 7', {'format': 'ixt5:num-dot-decimal', 'scale': '6', 'decimals': '-6'})

France: the ESEF report's facts as a table, and one of its HTML tables cell by cell:

In [4]:
report = Company("969500XXRPGD7HCAFA90", market="FR").filings(form="annual").latest()
frame = report.facts_to_pandas()
frame[frame["concept_raw"] == "ifrs-full:Revenue"][["value", "unit", "period_end"]].head()

,value,unit,period_end
0,"9 480,6",iso4217:EUR,2025-12-31
1,"8 648,9",iso4217:EUR,2024-12-31


HTML tables come back cell by cell as filed, with no inferred column alignment:

In [5]:
tenk = Company("AAPL", market="US").filings(form="annual").latest()
table = next(table for table in tenk.tables()
             if any("Total net sales" in cell.text for row in table.rows for cell in row))
[[cell.text for cell in row if cell.text] for row in table.rows[:8]]

[[], ['2025', 'Change', '2024', 'Change', '2023'], ['Americas', '$', '178,353', '7', '%', '$', '167,045', '3', '%', '$', '162,560'], ['Europe', '111,032', '10', '%', '101,328', '7', '%', '94,294'], ['Greater China', '64,377', '(4)', '%', '66,952', '(8)', '%', '72,559'], ['Japan', '28,703', '15', '%', '25,052', '3', '%', '24,257'], ['Rest of Asia Pacific', '33,696', '10', '%', '30,658', '4', '%', '29,615'], ['Total net sales', '$', '416,161', '6', '%', '$', '391,035', '2', '%', '$', '383,285']]

The ESEF index (limited tier) gives several years of iXBRL reports by LEI, newest
report period first. It lags the regulators by months, so it is history, not the
latest report.

In [6]:
enel = Company("WOCMU6HCI0OJWNPRZS33", market="IT", source="esef_index")
[(filing.period_end, filing.language) for filing in enel.filings(form="annual").head(4)]

[(datetime.date(2024, 12, 31), 'en'), (datetime.date(2024, 12, 31), 'it'), (datetime.date(2023, 12, 31), 'en'), (datetime.date(2023, 12, 31), 'it')]

In [7]:
print(provider_info("IT", source="esef_index").coverage_note)

Secondary index of ESEF annual reports by LEI; it lags regulators by months and is incomplete. On 2026-10-08 it held 974 FY2025 December-year-end reports against 2,142 for FY2024 (45%), with whole countries at 0%.
